# Robustness of numerical integration

In [ ]:
#    APM41012EP course notebook - Chapter 7 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Limits of Euler's method
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import root
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Explicit Euler

The explicit Euler method to solve 

$${\mathrm d}_t y(t) = f(t,u)$$

with $y(0)=y_0$ reads:

$$
\left\{
\begin{aligned}
& y^0 = y_0 \\
& y^{n+1} = y^n + \Delta t \; f(t^n,y^n) \quad \text{where} \quad \Delta t = t^{n+1} - t^n
\end{aligned}
\right.
$$

In [ ]:
class ode_result:
    def __init__(self, y, t):
        self.y = y
        self.t = t

def forward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y[:,it+1] = yn + dt*np.atleast_1d(fcn(tn, yn))

    return ode_result(y, t)

## Implicit Euler

The implicit Euler method to solve 

$${\mathrm d}_t y(t) = f(t,u)$$

with $y(0)=y_0$ reads:

$$
\left\{
\begin{aligned}
& y^0 = y_0 \\
& y^{n+1} = y^n + \Delta t \; f(t^{n+1},U^{n+1}) \quad \text{where} \quad \Delta t = t^{n+1} - t^n
\end{aligned}
\right.
$$

In [ ]:
def backward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt), order='F')
    y[:,0] = yini

    def g(uip1, *args):
        uip, tip1 = args
        return uip1 - uip - dt*np.array(fcn(tip1, uip1))

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y0 = yn + dt*np.atleast_1d(fcn(tn, yn))
        # solve y[:,it+1] - y[:,it] - dt * fcn(tini + (it+1)*dt, y[:,it+1]) = 0
        sol = root(g, y0, (yn, tn+dt))
        y[:,it+1] = sol.x

    return ode_result(y, t)

## Application to thermal explosion

We tackle the theory of thermal explosion studied by Semenov and Frank-Kamenetskii in the first part of the 20th century. We consider a homogeneous chemical reactor described by the evolution of a dimensionless temperature $\theta$. The variable $\theta$ is zero at the initial time, it is monotonically increasing and, following a thermal runaway, it will rise abruptly and reach the dimensionless adiabatic flame temperature $T_r$ at the end of the reaction. Under some assumptions on the reaction (rate constant in the form of an Arrhenius law, global reaction of order one with respect to the fuel mass fraction, non-dimensionalisation of time so that the ignition time is around
1 - large activation energy), which we will not detail, one arrives at a differential equation of the following form:

$$
d_{\tau} \widetilde{\theta} = f(\widetilde{\theta}) = \exp\Bigg(\frac{\widetilde{\theta}}{1 + \widetilde{\theta} \, / \, \beta}\Bigg) \Bigg(1 - \frac{\widetilde{\theta}}{\overline T_r} \Bigg)
$$

where $\beta$ is directly related to the activation energy ($\beta = \frac{E_a}{RT_0}$, where $E_a$ is the activation energy of the reaction, $R$ the ideal gas constant and $T_0$ the dimensional initial temperature).

In [ ]:
class explosion_with_consumption_1eq_model:

    def __init__(self, Tr, beta):
        self.Tr = Tr
        self.beta = beta 

    def fcn(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        theta_dot = np.exp(theta/(1+(theta/beta))) * (1-theta/Tr) 
        return theta_dot
    
    def jac(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        exp = np.exp(theta/(1+(theta/beta)))
        return (exp/(1+theta/beta)**2) * (1-theta/Tr) - exp/Tr
    
def plot_sol(Tr, beta, tini, tend, nt, yini):
    
    emwc1eq = explosion_with_consumption_1eq_model(Tr, beta)
    fcn = emwc1eq.fcn
    jac = emwc1eq.jac

    fig = go.Figure()

    for i, nt_i in enumerate(nt):

        dt = (tend-tini) / (nt_i-1)

        emwc1eq = explosion_with_consumption_1eq_model(Tr, beta)
        fcn = emwc1eq.fcn
        jac = emwc1eq.jac

        sol_ref = solve_ivp(fcn, (tini, tend), yini, method='Radau', atol=1.e-8, rtol=1.e-8)
        sol_fe = forward_euler(tini, tend, nt_i, yini, fcn)
        sol_be = backward_euler(tini, tend, nt_i, yini, fcn)

        fig.add_trace(go.Scatter(visible=False, x=sol_ref.t, y=sol_ref.y[0],  line_color='rgb(76,114,176)', mode='lines+markers',
                              name='ref sol'))
        fig.add_trace(go.Scatter(visible=False, x=sol_fe.t, y=sol_fe.y[0], line_color='rgb(221,132,82)', mode='lines+markers',
                             name='forward euler'))
        fig.add_trace(go.Scatter(visible=False, x=sol_be.t, y=sol_be.y[0], line_color='rgb(85,168,104)', mode='lines+markers',
                             name='backward euler'))

    for i in range(9,12): fig.data[i].visible = True

    # create slider
    steps = []
    for i, nt_i in enumerate(nt):
        step = dict(method="update", label = f"{nt_i}", 
                    args=[{"visible": [(el==3*i) or (el==3*i+1) or (el==3*i+2) for el in range(len(fig.data))]}])
        steps.append(step)
    sliders = [dict(currentvalue={'prefix': 'nt = '}, steps=steps, active=3)]

    fig.update_layout(sliders=sliders, height=500, legend=dict(orientation="h", y=1.1), title="Numerical integration")
    fig.show()

In [ ]:
Tr = 200.0
beta = 10. 

tini = 0.0
tend = 5.0
nt = np.array([171, 201, 401, 501])

yini = (0.,)

plot_sol(Tr, beta, tini, tend, nt, yini)